### Launch the ROS demo

In [ ]:
%%bash --bg
trap 'kill $(jobs -p)' EXIT
xterm -e /bin/bash -l -c "ros2 launch lecture demo.launch.xml"

Starting job # 0 in a separate thread.


### Python Code

In [1]:
# Import rclpy and create a ROS2 node
import rclpy
from rclpy.node import Node

rclpy.init()
node = Node('mynode')

In [ ]:
# Import and instantiate the robot model
from robot_model import RobotModel
robot = RobotModel(node)

In [ ]:
# Create a publisher and JointState message
from sensor_msgs.msg import JointState
pub = node.create_publisher(JointState, '/target_joint_states', 1)
msg = JointState()

In [ ]:
# Create a list of slider widgets, one for each joint
import ipywidgets
from ipywidgets import FloatSlider, Layout, Button, Box
joint_widgets = [FloatSlider(min = j.min, max = j.max, step = (j.max-j.min) / 100, description = j.name) \
                 for j in robot.active_joints]

In [ ]:
# Define a callback function to compute the forward-kinematics and publish a frame marker
from markers import frame, MarkerArray
marker_pub = node.create_publisher(MarkerArray, '/marker_array', 1)

def publish_fk_marker():
    T, _ = robot.fk(link='panda_link8', joints={j.description: j.value for j in joint_widgets})
    marker_pub.publish(frame(T))
    rclpy.spin_once(node, timeout_sec=0.0)

In [ ]:
# React to slider (value) changes by publishing this particular joint as well as the updated FK marker
def on_sent(event):
    widget = event.owner
    msg.name = [widget.description]
    msg.position = [widget.value]
    pub.publish(msg)
    publish_fk_marker()

for widget in joint_widgets:
    widget.observe(on_sent, 'value')

In [ ]:
# Create a button to randomly generate a pose within joint limits
def set_joints(values):
    for widget, value in values:
        widget.unobserve(on_sent, 'value')
        widget.value = value
        widget.observe(on_sent, 'value')
    msg.name, msg.position = map(list, zip(*[(widget.description, widget.value) for widget in joint_widgets]))
    pub.publish(msg)
    publish_fk_marker()

import random
def on_randomize(randomize):
    set_joints([(widget, random.uniform(widget.min, widget.max)) for widget in joint_widgets])

randomize = Button(description='Randomize')
randomize.on_click(on_randomize)

In [ ]:
# Create a button to center all joints within their limits
def on_center(b):
    set_joints([(widget, (widget.min + widget.max) / 2) for widget in joint_widgets])

center = Button(description='Center')
center.on_click(on_center)

In [ ]:
# Collect all widgets (sliders and buttons) in a form and display them
form_items = list(joint_widgets)
form_items += [Box([center, randomize])]

form = Box(form_items, layout=Layout(
    display='flex',
    flex_flow='column',
    border='solid 2px',
    align_items='stretch',
    width='100%'
))
display(form)

In [ ]:
# Optional cleanup when you are done with the notebook:
## node.destroy_node()
## rclpy.shutdown()